# Notebook 02 — Anomaly Detection
## Project: Medicare Provider Billing Anomalies in No-Fault PIP States

**This notebook:**
1. Runs Isolation Forest to score every provider by anomaly level
2. Checks that results are stable across two random seeds
3. Runs K-means clustering to characterize fraud types
4. Cross-references flagged providers against the OIG exclusion list
5. Produces the flagged provider table and two charts

**Input:**  `data/processed/cms_features.parquet`

**Outputs:**
- `outputs/figures/02_anomaly_scatter.png`
- `outputs/figures/03_cluster_profiles.png`
- `outputs/reports/flagged_providers.csv`

---

## 0. Setup

In [ ]:
import subprocess, sys
for pkg in ['scikit-learn', 'pandas', 'numpy', 'matplotlib', 'requests']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('All packages ready.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

os.makedirs('outputs/figures',  exist_ok=True)
os.makedirs('outputs/reports',  exist_ok=True)
os.makedirs('data/raw',         exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'font.size': 11,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white'
})

FRAUD_COLOR  = '#D85A30'
LEGIT_COLOR  = '#378ADD'
NEUT_COLOR   = '#888780'
WARN_COLOR   = '#EF9F27'
PURPLE_COLOR = '#7F77DD'

CLUSTER_COLORS = [FRAUD_COLOR, LEGIT_COLOR, WARN_COLOR, PURPLE_COLOR]

print('Setup complete.')

In [ ]:
# Load provider features
df = pd.read_parquet('data/processed/cms_features.parquet')

print(f'Loaded: {df.shape}')
print(f'Unique providers: {df["Rndrng_NPI"].nunique():,}')
print(f'Columns: {df.columns.tolist()}')

---
## 1. Prepare features for modeling

Isolation Forest and K-means both use numeric features only.  
We use four features — the two raw billing metrics and their z-scores.  
Using both raw values and z-scores gives the model two perspectives:  
absolute billing level AND relative standing among peers.

In [ ]:
# Features for anomaly detection
MODEL_FEATURES = [
    'median_inflation_ratio',    # how inflated is billing in absolute terms
    'median_svc_per_patient',    # how many services per patient
    'max_inflation_zscore',      # how anomalous vs specialty peers
    'max_svc_zscore',            # how anomalous services/patient vs peers
]

# Exclude Hawaii low-confidence providers from modeling
# Keep them in the dataset but don't let them influence the model
df_model = df[df['low_confidence'] == False].copy()
df_hi    = df[df['low_confidence'] == True].copy()

print(f'Providers in model:          {len(df_model):,}')
print(f'Hawaii providers (excluded): {len(df_hi):,}')

X = df_model[MODEL_FEATURES].copy()

# Fill any remaining nulls with 0
X = X.fillna(0)

# Cap extreme outliers at 99.5th percentile
# Extreme values skew distance calculations in clustering
for col in X.columns:
    cap = X[col].quantile(0.995)
    X[col] = X[col].clip(upper=cap)

# Scale features — both models are distance-sensitive
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nFeature matrix: {X_scaled.shape}')
print(f'NaNs after cleaning: {np.isnan(X_scaled).sum()}')

---
## 2. Isolation Forest

Isolation Forest finds anomalies by measuring how quickly each provider  
can be separated from the rest using random yes/no questions about their features.  
Providers that get isolated quickly are anomalous — they answer questions  
very differently from the crowd.

**contamination=0.05** means we tell the model to treat the top 5% most  
isolated providers as anomalies. In a PIP fraud context 5% is conservative —  
it surfaces the clearest cases without over-flagging.

In [ ]:
print('Running Isolation Forest...')

iso_forest = IsolationForest(
    n_estimators=100,     # 100 random trees
    contamination=0.05,   # expect ~5% anomalous providers
    random_state=42,      # fixed seed — makes results reproducible
    n_jobs=-1
)

iso_forest.fit(X_scaled)

# decision_function: more negative = more anomalous
# We flip the sign so higher score = more suspicious
raw_scores = iso_forest.decision_function(X_scaled)
df_model['anomaly_score'] = -raw_scores

# is_anomaly flag: -1 from Isolation Forest = anomalous
# We convert to 1=anomalous, 0=normal for readability
predictions = iso_forest.predict(X_scaled)
df_model['is_anomaly'] = (predictions == -1).astype(int)

n_anomalous = df_model['is_anomaly'].sum()
print(f'\nProviders flagged as anomalous: {n_anomalous:,} '
      f'({n_anomalous/len(df_model)*100:.1f}% of total)')
print(f'\nAnomaly score distribution:')
print(df_model['anomaly_score'].describe().round(3).to_string())

---
## 3. Anomaly score stability check

Isolation Forest is stochastic — it uses randomness to build trees.  
We run it twice with different starting seeds and check whether  
the same providers appear at the top both times.

High overlap = results are real signal, not a random accident.  
Low overlap = borderline cases are sensitive to random variation.

In [ ]:
# Run 1: seed 42 (already done above)
top20_seed42 = set(
    df_model.nlargest(20, 'anomaly_score')['Rndrng_NPI'].astype(str)
)

# Run 2: seed 99 — different random starting point
iso_seed99 = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=99,     # different seed
    n_jobs=-1
)
iso_seed99.fit(X_scaled)
scores_seed99 = -iso_seed99.decision_function(X_scaled)
df_model['anomaly_score_seed99'] = scores_seed99

top20_seed99 = set(
    df_model.nlargest(20, 'anomaly_score_seed99')['Rndrng_NPI'].astype(str)
)

# Measure overlap
overlap = top20_seed42 & top20_seed99
overlap_count = len(overlap)

print('=== Anomaly score stability check ===')
print(f'Top 20 providers — seed 42:  {len(top20_seed42)}')
print(f'Top 20 providers — seed 99:  {len(top20_seed99)}')
print(f'Overlap (same providers):    {overlap_count} of 20')
print(f'Stability rate:              {overlap_count/20*100:.0f}%')
print()

if overlap_count >= 16:
    print('Result: HIGH STABILITY')
    print('80%+ of the same providers flagged under both seeds.')
    print('Findings are robust to random variation.')
elif overlap_count >= 12:
    print('Result: MODERATE STABILITY')
    print('Some providers near the boundary are seed-sensitive.')
    print('Top 10 results are reliable; positions 11-20 should be noted cautiously.')
else:
    print('Result: LOW STABILITY')
    print('Results vary significantly across seeds.')
    print('Consider increasing n_estimators to 200 for more stable results.')

# Drop the second score — use seed 42 as canonical
df_model.drop(columns=['anomaly_score_seed99'], inplace=True)

---
## 4. K-means clustering

Isolation Forest tells us WHO is anomalous.  
K-means clustering tells us WHAT TYPE of anomaly they are.

Three clusters capture the main fraud patterns seen in PIP billing:
- High inflation, moderate volume — inflated charges per service
- High volume, high services per patient — treatment mill pattern  
- Extreme on both dimensions — organized fraud ring pattern

In [ ]:
N_CLUSTERS = 3

kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=42,
    n_init=10     # run 10 times with different starting points, keep best
)
df_model['cluster'] = kmeans.fit_predict(X_scaled)

# Profile each cluster by its average feature values
cluster_profile = df_model.groupby('cluster')[MODEL_FEATURES + ['is_anomaly']].mean().round(2)

print('=== Cluster profiles ===')
print(cluster_profile.to_string())
print(f'\nCluster sizes:')
print(df_model['cluster'].value_counts().sort_index().to_string())

In [ ]:
# Assign domain labels to clusters based on their profiles
# The cluster with highest median_inflation_ratio is the billing inflation cluster
# The cluster with highest median_svc_per_patient is the treatment mill cluster
# The cluster with both high is the organized fraud cluster

inflation_ranks = cluster_profile['median_inflation_ratio'].rank(ascending=False)
svc_ranks       = cluster_profile['median_svc_per_patient'].rank(ascending=False)
combined_rank   = (inflation_ranks + svc_ranks)

cluster_labels = {}
for cluster_id in range(N_CLUSTERS):
    inf_rank = inflation_ranks[cluster_id]
    svc_rank = svc_ranks[cluster_id]

    if combined_rank[cluster_id] == combined_rank.min():
        cluster_labels[cluster_id] = 'High inflation + high volume'
    elif inf_rank < svc_rank:
        cluster_labels[cluster_id] = 'Billing inflation'
    else:
        cluster_labels[cluster_id] = 'Treatment mill'

df_model['cluster_label'] = df_model['cluster'].map(cluster_labels)

print('Cluster labels assigned:')
for k, v in cluster_labels.items():
    size = (df_model['cluster'] == k).sum()
    anomaly_pct = df_model[
        df_model['cluster'] == k
    ]['is_anomaly'].mean() * 100
    print(f'  Cluster {k}: {v} | n={size:,} | {anomaly_pct:.1f}% flagged anomalous')

---
## 5. OIG exclusion list cross-reference

The Office of Inspector General publishes a list of providers  
excluded from Medicare for fraud and abuse. It is publicly available  
and downloadable for free.

If our unsupervised model independently surfaces providers that  
regulators already sanctioned, that is meaningful external validation —  
two independent processes reaching the same conclusion.

In [ ]:
# Download OIG exclusion list
# Published at: https://oig.hhs.gov/exclusions/downloadables.asp
# Updated monthly — free public dataset

import requests

OIG_URL  = 'https://oig.hhs.gov/exclusions/downloadables/UPDATED.csv'
OIG_PATH = 'data/raw/oig_exclusions.csv'

if not os.path.exists(OIG_PATH):
    print('Downloading OIG exclusion list...')
    try:
        response = requests.get(OIG_URL, timeout=30)
        with open(OIG_PATH, 'wb') as f:
            f.write(response.content)
        print(f'Downloaded: {os.path.getsize(OIG_PATH)/1e6:.1f} MB')
    except Exception as e:
        print(f'Download failed: {e}')
        print('Manual download: https://oig.hhs.gov/exclusions/downloadables.asp')
        print('Save as: data/raw/oig_exclusions.csv')
else:
    print(f'OIG file already exists: {os.path.getsize(OIG_PATH)/1e6:.1f} MB')

In [ ]:
# Cross-reference flagged providers against OIG exclusions
oig_matched = False

if os.path.exists(OIG_PATH):
    try:
        oig = pd.read_csv(OIG_PATH, dtype=str, encoding='latin-1')
        print(f'OIG exclusions loaded: {len(oig):,} records')
        print(f'OIG columns: {oig.columns.tolist()}')

        # NPI column may be named differently — find it
        npi_col = next(
            (c for c in oig.columns if 'NPI' in c.upper()), None
        )

        if npi_col:
            # Get our top 50 flagged providers
            top50_npis = set(
                df_model.nlargest(50, 'anomaly_score')['Rndrng_NPI']
                .astype(str)
            )
            oig_npis = set(oig[npi_col].dropna().astype(str))
            matches  = top50_npis & oig_npis

            print(f'\n=== OIG cross-reference results ===')
            print(f'Top 50 flagged providers checked: {len(top50_npis)}')
            print(f'Matches in OIG exclusion list:    {len(matches)}')

            if len(matches) > 0:
                oig_matched = True
                print(f'\nValidation: {len(matches)} provider(s) flagged by our model')
                print(f'were independently excluded by OIG for fraud/abuse.')
                print(f'This confirms the anomaly detection is finding real signal.')
                matched_detail = df_model[
                    df_model['Rndrng_NPI'].astype(str).isin(matches)
                ][['Rndrng_Prvdr_Last_Org_Name', 'Rndrng_Prvdr_Type',
                   'Rndrng_Prvdr_State_Abrvtn', 'anomaly_score',
                   'median_inflation_ratio']]
                print(matched_detail.to_string(index=False))
            else:
                print('No direct NPI matches found.')
                print('Note: OIG list may use name/address matching rather than NPI.')
                print('Stability check is the primary validation for this project.')
        else:
            print('NPI column not found in OIG file — skipping cross-reference.')
    except Exception as e:
        print(f'OIG cross-reference failed: {e}')
        print('Continuing without OIG validation.')
else:
    print('OIG file not available — skipping cross-reference.')
    print('Primary validation: anomaly score stability check (section 3).')

---
## 6. Visualizations

In [ ]:
# Chart 1: Anomaly scatter plot
# Billing inflation ratio vs services per patient
# Colored by cluster, top 2% highlighted

# Cap axes for readability
plot_df = df_model[
    (df_model['median_inflation_ratio'] <= 20) &
    (df_model['median_svc_per_patient'] <= 40)
].copy()

fig, ax = plt.subplots(figsize=(11, 7))

# Plot all providers colored by cluster
for cluster_id, label in cluster_labels.items():
    mask  = plot_df['cluster'] == cluster_id
    color = CLUSTER_COLORS[cluster_id]
    ax.scatter(
        plot_df.loc[mask, 'median_inflation_ratio'],
        plot_df.loc[mask, 'median_svc_per_patient'],
        alpha=0.25, s=10, color=color, label=f'Cluster: {label}'
    )

# Highlight top 2% most anomalous
top2pct_threshold = plot_df['anomaly_score'].quantile(0.98)
top_flagged = plot_df[
    plot_df['anomaly_score'] >= top2pct_threshold
]
ax.scatter(
    top_flagged['median_inflation_ratio'],
    top_flagged['median_svc_per_patient'],
    alpha=0.9, s=50, color='black',
    zorder=5, label=f'Top 2% most anomalous (n={len(top_flagged):,})'
)

# Threshold lines
ax.axvline(
    x=3, color='green', linestyle=':',
    linewidth=1.5, alpha=0.7, label='Legitimate billing bound (3x)'
)
ax.axhline(
    y=15, color='green', linestyle=':',
    linewidth=1.5, alpha=0.7, label='Legitimate services/patient bound (15)'
)

# Label top-right quadrant
ax.text(
    16, 36,
    'High inflation\n+ high volume\n= investigation\npriority',
    fontsize=9, color=FRAUD_COLOR, ha='center',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
              edgecolor=FRAUD_COLOR, alpha=0.8)
)

ax.set_title(
    'Provider anomaly map — billing inflation vs treatment volume\n'
    'Top-right quadrant = highest investigation priority',
    fontweight='bold'
)
ax.set_xlabel('Median billing inflation ratio (submitted ÷ Medicare payment)')
ax.set_ylabel('Median services per patient')
ax.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('outputs/figures/02_anomaly_scatter.png')
plt.show()
print('Saved: outputs/figures/02_anomaly_scatter.png')

In [ ]:
# Chart 2: Cluster profiles
# Average feature values per cluster — what makes each cluster distinct

profile = df_model.groupby('cluster_label')[MODEL_FEATURES].mean()

# Normalize each feature to 0-1 for visual comparison
profile_norm = (profile - profile.min()) / (profile.max() - profile.min())

feature_labels = [
    'Billing\ninflation ratio',
    'Services\nper patient',
    'Inflation\nz-score',
    'Svc/patient\nz-score'
]

x     = np.arange(len(feature_labels))
width = 0.25
clusters = profile_norm.index.tolist()

fig, ax = plt.subplots(figsize=(11, 5))

for i, (cluster_name, row) in enumerate(profile_norm.iterrows()):
    bars = ax.bar(
        x + i * width,
        row.values,
        width,
        label=cluster_name,
        color=CLUSTER_COLORS[i],
        alpha=0.8,
        edgecolor='none'
    )

ax.set_title(
    'Cluster profiles — what makes each fraud type distinct\n'
    'Height = relative feature level (normalized to 0-1 for comparison)',
    fontweight='bold'
)
ax.set_xticks(x + width)
ax.set_xticklabels(feature_labels)
ax.set_ylabel('Normalized feature value')
ax.legend(fontsize=9)
ax.set_ylim([0, 1.15])

plt.tight_layout()
plt.savefig('outputs/figures/03_cluster_profiles.png')
plt.show()
print('Saved: outputs/figures/03_cluster_profiles.png')

---
## 7. Flagged provider table

Top 15 most anomalous providers with plain-English domain flags.  
This is the operational output — what an SIU analyst would act on.

In [ ]:
def domain_flag(row):
    """
    Translate statistical signals into plain-English fraud indicators.
    Based on PIP claims examination experience.
    """
    flags = []

    if row['median_inflation_ratio'] > 10:
        flags.append(f'Bills {row["median_inflation_ratio"]:.0f}x Medicare rate')
    elif row['median_inflation_ratio'] > 5:
        flags.append(f'Bills {row["median_inflation_ratio"]:.0f}x Medicare rate')

    if row['median_svc_per_patient'] > 30:
        flags.append(
            f'{row["median_svc_per_patient"]:.0f} sessions/patient '
            f'(clinical norm: 8-14)'
        )
    elif row['median_svc_per_patient'] > 20:
        flags.append(
            f'{row["median_svc_per_patient"]:.0f} sessions/patient '
            f'(elevated)'
        )

    if row['max_inflation_zscore'] > 3:
        flags.append(
            f'Billing {row["max_inflation_zscore"]:.1f} SDs above specialty peers'
        )

    if row['is_organization'] == 1:
        flags.append('Organization entity (clinic/LLC)')

    return ' | '.join(flags) if flags else 'Anomalous pattern'


# Build top 15 flagged provider table
# Exclude Hawaii low-confidence providers
top15 = (
    df_model[
        df_model['low_confidence'] == False
    ]
    .nlargest(15, 'anomaly_score')
    .copy()
)

top15['domain_flags'] = top15.apply(domain_flag, axis=1)

output_cols = [
    'Rndrng_Prvdr_Last_Org_Name',
    'Rndrng_Prvdr_Type',
    'Rndrng_Prvdr_State_Abrvtn',
    'is_organization',
    'anomaly_score',
    'median_inflation_ratio',
    'median_svc_per_patient',
    'max_inflation_zscore',
    'cluster_label',
    'domain_flags'
]

top15_display = top15[output_cols].round(2)
top15_display.columns = [
    'Provider', 'Specialty', 'State', 'Org',
    'Anomaly Score', 'Inflation Ratio',
    'Svc/Patient', 'Inflation Z-Score',
    'Cluster', 'Domain Flags'
]

print('=== Top 15 Most Anomalous Providers ===')
print(top15_display.to_string(index=False))

In [ ]:
# Save flagged provider table
top15_display.to_csv(
    'outputs/reports/flagged_providers.csv',
    index=False
)
print('Saved: outputs/reports/flagged_providers.csv')

---
## Notebook 02 Summary

In [ ]:
print('=' * 60)
print('NOTEBOOK 02 COMPLETE — Anomaly Detection Summary')
print('=' * 60)
print(f'\nProviders analyzed:         {len(df_model):,}')
print(f'Flagged as anomalous (5%):  {df_model["is_anomaly"].sum():,}')
print(f'\nStability check:')
print(f'  Top 20 overlap across seeds: {overlap_count}/20 ({overlap_count/20*100:.0f}%)')
print(f'\nClusters identified: {N_CLUSTERS}')
for k, v in cluster_labels.items():
    n = (df_model['cluster'] == k).sum()
    anom_pct = df_model[df_model['cluster']==k]['is_anomaly'].mean()*100
    print(f'  Cluster {k}: {v} — {n:,} providers, {anom_pct:.1f}% anomalous')
print(f'\nOIG cross-reference: {"Matches found" if oig_matched else "No matches / file unavailable"}')
print(f'\nOutputs:')
print(f'  outputs/figures/02_anomaly_scatter.png')
print(f'  outputs/figures/03_cluster_profiles.png')
print(f'  outputs/reports/flagged_providers.csv')
print(f'\nKey finding for README:')
print(f'  {df_model["is_anomaly"].sum():,} of {len(df_model):,} providers '
      f'({df_model["is_anomaly"].mean()*100:.1f}%) show billing patterns')
print(f'  consistent with PIP fraud across {len(cluster_labels)} distinct fraud typologies.')